# TGT

# TGT Backend Architecture – Developer Guide

This document explains how TGT works and how you can use and extend it from a developer perspective.

It focuses on adapting the **backend logic and linguistic processing pipeline**, not the frontend.

The frontend was almost entirely built using **Builder.io**  
https://www.builder.io/

You can adapt the frontend using Builder, but **any frontend change must respect the backend API contracts**.

In particular, you must consider:

- The API calls in `frontend/src/hooks`
- The pages using those hooks in `frontend/src/pages`
- The backend responses defined in:
  - `backend/routers`
  - `backend/app`

> All API logic, including online workers, lives in `backend/routers`.

---

## Backend Architecture

The backend is structured hierarchically using **inheritance** to ensure compatibility between components.

```

backend/
├── inference/
│   ├── worker/
│   │   └── (local + abstract workers)
│   ├── processors/
│   ├── transcription/
│   ├── translation/
│   ├── glossing/
│   └── pii_strategy/
├── training/
├── materials/
│   └── secrets.env
└── utils/

```

> **Important:**  
> `materials/secrets.env` must contain all access tokens  
> (Azure, DeepL, HuggingFace, etc.).

All components are implemented as **classes**, except utility functions inside `utils`.

- **Use inheritance to define components**
- **Use factories to instantiate components**

---

## Core Components

### Worker

**Role:**  
Orchestrates the job.

- Accepts arguments
- Builds a job
- Creates a processor via `processor_factory`
- Runs the processor

**Type:**  
All workers are instances of `AbstractInferenceWorker`.

---

### Processor

**Role:**  
Executes the job logic.

- Instantiates a strategy using `strategy_factory`
- Binds:
  - language (e.g. German)
  - task (e.g. transcribe)
  - instruction (e.g. corrected_transcription)
- Defines:
  - directory traversal
  - file handling
  - logging logic

**Type:**  
All processors extend the abstract `DataProcessor`.

---

### Strategy

**Role:**  
Defines how a task is performed for a specific language and provider.

Examples:

- How do we transcribe Vietnamese?  
  → Use **PhoWhisper**
- How do we translate German?  
  → Use **DeepL**

Strategies are instances of:

- `TranscriptionStrategy`
- `TranslationStrategy`
- `GlossingStrategy`
- `PIIStrategy`

Each strategy implements the logic for:
- a **language**
- a **task**
- a **provider/model**

---

## Abstract Classes

Abstract base classes define **contracts** between layers.

They ensure:
- processors can safely use strategies
- workers can safely use processors

This guarantees compatibility across the entire pipeline.


# How the Application Runs

The application consists of two main components:

- **Frontend**: built in **TypeScript**
- **Backend**: built in **Python (FastAPI)**

The frontend is compiled into static files and automatically served by the backend.

---

## Frontend Build

To generate the frontend distribution, run:

```bash
npm run build
```

This creates a `dist/` directory containing the compiled frontend assets.

The backend (in `app.py`) is configured to **serve these static files directly**, so no separate frontend server is required in production.

---

## Backend Execution

To run the backend, navigate to the `backend/` directory and execute:

```bash
python -m uvicorn app:app --host 127.0.0.1 --port 8000 --reload
```

### What this command does

- `python -m uvicorn` → runs the Uvicorn ASGI server  
- `app:app` → loads the FastAPI instance named `app` from `app.py`  
- `--host 127.0.0.1` → binds to localhost  
- `--port 8000` → exposes the app on port 8000  
- `--reload` → automatically restarts the server on code changes  

Once running, the application is available at:

```
http://127.0.0.1:8000
```

---

## Alternative (FastAPI CLI)

You may also run the server using FastAPI’s built-in CLI:

```bash
pip install "fastapi[standard]"
fastapi run app
```

This uses Uvicorn internally and provides the same behavior with a simpler command.


# Let's create a strategy for transcribing Thai!!

1. First we create a new thai.py file under backend/inference/transcription to define our new thai transcription strategy. 
2. When inside backend/inference/transcription we call the abstract TranscriptionStrategy class and initiate our ThaiTranscriptionStrategy as an instance of the abstract TranscriptionStrategy and copy the abstract methods that need to be instantiated


In [ ]:
from inference.transcription.abstract import TranscriptionStrategy

class ThaiStrategy(TranscriptionStrategy):
    def __init__(self, language_code, device = "cpu"):
        super().__init__(language_code, device)

    def load_model(self):
       pass
        
    def transcribe(self, path_to_audio: str) -> str | None:
        pass

ModuleNotFoundError: No module named 'inference'

Now we look for an appropriate model. You can look for models in the HuggingFace hub.

Let's use this https://github.com/biodatlab/thonburian-whisper

Follow the instructions on how to use the model and adjust the calls

In [ ]:
from inference.transcription.abstract import TranscriptionStrategy
import torch
from transformers import pipeline

class ThaiStrategy(TranscriptionStrategy):
    def __init__(self, language_code, device = "cpu"):
        super().__init__(language_code, device)

    def load_model(self):
        MODEL_NAME = "biodatlab/whisper-th-medium-combined"  # see alternative model names below
        device = 0 if torch.cuda.is_available() else "cpu"
        self.pipe = pipeline(
            task="automatic-speech-recognition",
            model=MODEL_NAME,
            chunk_length_s=30,
            device=device,
        )
        
    def transcribe(self, path_to_audio: str) -> str | None:
        out = self.pipe(path_to_audio, generate_kwargs={"language":"<|th|>", "task":"transcribe"}, batch_size=16)["text"]
        return out

Now we need to modify the factory to make sure this strategy is called when thai is the specified language

In [ ]:
from inference.transcription.abstract import TranscriptionStrategy
from inference.transcription.whisperx import WhisperxStrategy
from inference.transcription.whisper import WhisperStrategy
from inference.transcription.bengali import BengaliStrategy
from inference.transcription.vietnamese import VietnameseStrategy
from inference.transcription.thai import ThaiStrategy #this is new




class TranscriptionStrategyFactory:
    @staticmethod
    def get_strategy(language_code: str) -> TranscriptionStrategy:
        if language_code in ['en', 'fr', 'de', 'es', 'it', 'ja', 'zh', 'nl',
                             'uk', 'pt', 'ar', 'cs', 'ru', 'pl', 'hu', 'fi',
                             'fa', 'el', 'tr', 'da', 'he', 'ko', 'ur', 
                             'te', 'hi', 'ca', 'ml', 'ro', 'ka', 'tl']:
            return WhisperxStrategy(language_code)
        elif language_code in ['vi']:
            return VietnameseStrategy(language_code)
        elif language_code in ['et']:
            return WhisperStrategy(language_code)
        elif language_code in ['bn']:
            return BengaliStrategy(language_code)
        elif language_code in ['th']: # this is also new
            return ThaiStrategy(language_code) 
        else:
            raise ValueError(f"No transcription strategy available for language code: {language_code}")


It is done! Now lets call the app and see how it works!

Important notes:

- If some library is not included, you need to be sure to install libraries and be careful of potential conflicts.
- Note that this is retrieving the model from huggingface and therefore you need to have an HuggingFace token under secrets
